# Chapter 5: Writing Your First Complete Python Script

Companion notebook to **python-from-zero** · `lesson-05` · based on *Python for VLSI*, Chapter 5.

### You will learn

- Structure a real script with imports, functions, and a `main`.
- Read inputs, compute, and write outputs cleanly.
- Use `if __name__ == '__main__':` to stay importable.
- Run the script both inline and via `runpy`.

In [1]:
import sys, platform
print(f"Python {sys.version.split()[0]} on {platform.system()}")
assert sys.version_info >= (3, 9), "This notebook needs Python 3.9 or newer."

Python 3.12.3 on Linux


## Script anatomy

A good script starts with a module docstring, imports, helper functions, and a `main()` guarded by `if __name__ == '__main__':`.

In [2]:
from pathlib import Path

Path("power_calc.py").write_text('''"""Compute dynamic power for a tiny cell list."""
from pathlib import Path

def dynamic_power(cap_pf, freq_ghz, v_v):
    # P = C * V^2 * f
    return cap_pf * 1e-12 * v_v**2 * freq_ghz * 1e9

def main():
    cells = [("INV_X1", 0.80, 1.0, 0.9),
             ("NAND2_X1", 1.20, 1.0, 0.9),
             ("DFF_X1", 5.00, 1.0, 0.9)]
    total = 0.0
    for name, cap, f, v in cells:
        p = dynamic_power(cap, f, v)
        total += p
        print(f"{name:10s} P={p*1e3:.3f} mW")
    print(f"total = {total*1e3:.3f} mW")

if __name__ == "__main__":
    main()
''')
print(Path("power_calc.py").read_text()[:200], "...")


"""Compute dynamic power for a tiny cell list."""
from pathlib import Path

def dynamic_power(cap_pf, freq_ghz, v_v):
    # P = C * V^2 * f
    return cap_pf * 1e-12 * v_v**2 * freq_ghz * 1e9

def mai ...


## Running the script in-process

`runpy.run_path` executes a script as if it were `python power_calc.py`.

In [3]:
import runpy
runpy.run_path("power_calc.py", run_name="__main__")


INV_X1     P=0.648 mW
NAND2_X1   P=0.972 mW
DFF_X1     P=4.050 mW
total = 5.670 mW


{'__name__': '__main__',
 '__doc__': 'Compute dynamic power for a tiny cell list.',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'power_calc.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, locals=None, fromli

## Importing functions from the script

Because of the `__main__` guard the script is safe to import without running `main()`.

In [4]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location("power_calc", "power_calc.py")
mod = importlib.util.module_from_spec(spec)
sys.modules["power_calc"] = mod
spec.loader.exec_module(mod)

print("imported fn:", mod.dynamic_power.__name__)
print("sample     :", round(mod.dynamic_power(1.5, 1.0, 0.9) * 1e3, 4), "mW")


imported fn: dynamic_power
sample     : 1.215 mW


## Reading a simple input file

Cells pairs `name, cap_pf` one per line. Script parses and computes.

In [5]:
from pathlib import Path
Path("cells.csv").write_text(
    "INV_X1,0.80\n"
    "NAND2_X1,1.20\n"
    "DFF_X1,5.00\n"
    "BUF_X1,0.60\n"
)

rows = []
for line in Path("cells.csv").read_text().splitlines():
    if not line.strip():
        continue
    name, cap = line.split(",")
    rows.append((name, float(cap)))
print(rows)


[('INV_X1', 0.8), ('NAND2_X1', 1.2), ('DFF_X1', 5.0), ('BUF_X1', 0.6)]


## Writing an output file

Always use `with open(..., 'w')` so the file closes cleanly on error.

In [6]:
from pathlib import Path

summary = "\n".join(f"{name},{cap:.2f}" for name, cap in rows) + "\n"
Path("cells_clean.csv").write_text(summary)
print(Path("cells_clean.csv").read_text())


INV_X1,0.80
NAND2_X1,1.20
DFF_X1,5.00
BUF_X1,0.60



## Basic argument handling

Until we reach `argparse` in Ch. 14, we use a tiny hand-written loop so you see what a CLI boils down to.

In [7]:
import sys

def parse_argv(argv):
    opts = {"verbose": False, "input": None}
    it = iter(argv[1:])
    for token in it:
        if token == "--verbose":
            opts["verbose"] = True
        elif token == "-i":
            opts["input"] = next(it)
    return opts

print(parse_argv(["prog", "-i", "cells.csv", "--verbose"]))


{'verbose': True, 'input': 'cells.csv'}


## VLSI practice — end-to-end dynamic-power script

Read `cells.csv`, compute total dynamic power at 1 GHz and 0.9 V, and write `power_report.txt`.

In [8]:
from pathlib import Path

def p_dyn(cap_pf, f_ghz, v):
    return cap_pf * 1e-12 * v*v * f_ghz * 1e9

lines = []
total = 0.0
for row in Path("cells.csv").read_text().splitlines():
    if not row.strip():
        continue
    name, cap = row.split(",")
    p = p_dyn(float(cap), 1.0, 0.9)
    total += p
    lines.append(f"{name:10s} {p*1e3:.3f} mW")

lines.append(f"{'TOTAL':10s} {total*1e3:.3f} mW")
Path("power_report.txt").write_text("\n".join(lines) + "\n")
print(Path("power_report.txt").read_text())


INV_X1     0.648 mW
NAND2_X1   0.972 mW
DFF_X1     4.050 mW
BUF_X1     0.486 mW
TOTAL      6.156 mW



### Exercise 1
Write a 3-line script that prints its own `__name__`.

In [9]:
# Your code here


<details><summary>Show solution</summary>

```python
from pathlib import Path; Path('hi.py').write_text('print(__name__)\n')
import runpy; runpy.run_path('hi.py', run_name='__main__')
```

</details>

### Exercise 2
Given `cap = 1.2e-12`, `v = 0.9`, `f = 1e9`, compute P = C·V²·f in mW.

In [10]:
# Your code here


<details><summary>Show solution</summary>

```python
c=1.2e-12; v=0.9; f=1e9
print(round(c*v*v*f*1e3, 4), 'mW')
```

</details>

### Exercise 3
Write `cells.csv` rows into a dict `{name: cap}`.

In [11]:
# Your code here


<details><summary>Show solution</summary>

```python
from pathlib import Path
d={}
for r in Path('cells.csv').read_text().splitlines():
    if r: n,c=r.split(','); d[n]=float(c)
print(d)
```

</details>

## Recap

- Real scripts have imports, helpers, a `main`, and a `__main__` guard.
- Use `runpy.run_path` or `importlib` to run/import scripts in notebooks.
- Read inputs with `pathlib`, compute, then write outputs with `with`.
- Hand-rolled argv parsing reveals what `argparse` will automate later.

## What's next

Continue with **Chapter 6: Integer Mastery for VLSI Automation** in this repo, and the matching lesson on [python-from-zero](https://python-from-zero.pages.dev).